# Проект 1. Окупилось ли повышение цен в марте 2026

**Бизнес-вопрос.** С 1 марта 2026 ПРАЙМ поднял цены: "Базовый" с 199 до 249 ₽, "Расширенный" с 399 до 449 ₽.
Принесло ли это больше денег, или часть выигрыша съел отток? Что делать с ценами дальше?

**Гипотезы**, которые проверяем в ноутбуке:

1. Выручка после повышения выросла за счёт цены, а не за счёт базы.
2. После повышения вырос отток (клиенты ушли из-за цены).
3. Сильнее ушли клиенты дешёвого тарифа или отдельных каналов.

**Договорённости расчёта** (пишем их в подписи к каждому числу):

- время и месяцы считаем по **UTC**, как хранит база;
- выручка = **успешные платежи** (`status = 'success'`), возвраты и неудачные не берём;
- выручку берём из `payments` и **не соединяем** с участниками подписок: так нет двойного счёта из-за шеринга;
- месяцы разной длины сравниваем **в пересчёте на день**;
- клиентов без канала показываем отдельной строкой "без канала".

## 0. Подключение

Функция `q(sql)` отправляет запрос в базу и возвращает таблицу pandas. Пароля в ноутбуке нет:
его хранит файл `.env`, а читает `get_engine()` из `src/prime/config.py`.

In [ ]:
import math
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sqlalchemy import text

# Наши функции из репозитория: подключение, проверки, метрики, разложение.
try:
    from prime.config import get_engine
    from prime.checks import find_duplicate_payments, find_missing_channel
    from prime.metrics import active_on, churn_rate
    from prime.factors import chain, contrib, waterfall
except ModuleNotFoundError:
    # Понятная ошибка вместо длинного трейсбека: ноутбук открыт не в окружении проекта.
    raise ModuleNotFoundError(
        "Пакет prime не найден. Запустите Jupyter из корня репозитория командой: uv run jupyter lab\n"
        f"Сейчас Python отсюда: {sys.executable}"
    )

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить SQL-запрос и вернуть DataFrame. Соединение закрываем сразу."""
    try:
        with engine.connect() as conn:
            # Сутки и месяцы считаем по UTC: так они одинаковые у всех, кто запускает ноутбук.
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        # Сервер общий на всю группу, лишние соединения не держим.
        engine.dispose()


# Тысячи через пробел в выводе таблиц: так числа читать легче.
pd.set_option("display.float_format", lambda x: f"{x:,.2f}".replace(",", " "))

print("pandas", pd.__version__)
print("подключились как:", q("select current_user as login").iloc[0, 0])

## 1. Приёмка данных

Прежде чем что-то считать, задаём таблицам семь вопросов приёмки из Л3: сколько строк и сколько ждали,
какие типы, сколько весит, одна строка это кто, где пропуски, нет ли невозможных значений, нет ли дыр по датам.

Нужны три таблицы:

| Таблица | Зерно (одна строка это) | Зачем нам |
|---|---|---|
| `tariffs` | цена тарифа в интервале дат | узнать, когда и насколько подняли цены |
| `payments` | одна попытка списания | выручка и число платежей |
| `subscriptions` | одна подписка | активная база, отток, канал владельца |

### 1.1 Справочник цен

У тарифа несколько строк: цена меняется, и у каждой строки свой интервал `valid_from` ... `valid_to`.
На семинаре 4 в нём нашлась строка, которая перекрывает соседние. Проверяем это кодом, а не глазами.

In [ ]:
tariffs = q("select * from prime.tariffs order by tariff_id, valid_from")

print(tariffs.dtypes)
tariffs

In [ ]:
# Ищем пересечения интервалов самосоединением (Л4, слайд 14).
# Соединяем справочник сам с собой по tariff_id: получаем все пары строк одного тарифа.
a = tariffs.reset_index(names="row_id")
pairs = a.merge(a, on="tariff_id", suffixes=("_a", "_b"))

overlap = pairs[
    (pairs["row_id_a"] < pairs["row_id_b"])               # без этого каждая строка "пересечётся" сама с собой
    & (pairs["valid_from_a"] <= pairs["valid_to_b"])      # два интервала пересекаются, если каждый
    & (pairs["valid_from_b"] <= pairs["valid_to_a"])      # начинается не позже конца другого
]

print("пересекающихся пар:", len(overlap))
overlap[["tariff_id", "monthly_fee_a", "valid_from_a", "valid_to_a", "monthly_fee_b", "valid_from_b", "valid_to_b"]]

### 1.2 Платежи

Берём период с декабря 2025 по июнь 2026: два месяца до повышения цен, март и три месяца после.
Июль и август не берём: в конце августа случился обвал partner_telecom (Л6), он смешал бы два разных события.

Колонки выбираем в SQL, а не "выгружу всё, потом посмотрю": так кадр весит меньше (Л3, слайд 11).
Тариф берём из подписки: у платежа своего тарифа нет.

In [ ]:
PERIOD = ("2025-12-01", "2026-07-01")   # [начало, конец): правый конец не включён

# Контрольное число строк считаем в базе ДО выгрузки: потом сверим с тем, что приехало.
expected_rows = q("""
    select count(*) from prime.payments
    where paid_at >= :a and paid_at < :b
""", a=PERIOD[0], b=PERIOD[1]).iloc[0, 0]

pay = q("""
    select p.payment_id, p.subscription_id, p.paid_at, p.amount, p.status, s.tariff_id
    from prime.payments p
    join prime.subscriptions s on s.subscription_id = p.subscription_id
    where p.paid_at >= :a and p.paid_at < :b
""", a=PERIOD[0], b=PERIOD[1])

# Вопрос 1. Сколько строк и сколько ждали. inner join с подписками не должен потерять ни одной строки:
# у каждого платежа есть подписка.
print("1. строк:", len(pay), " ждали:", expected_rows, "->", "ok" if len(pay) == expected_rows else "ВНИМАНИЕ")
# Вопрос 2. Какие типы выбрал за нас pandas.
print("\n2. типы:"); print(pay.dtypes)
# Вопрос 3. Сколько весит. deep=True обязателен: без него pandas не считает сами строки текста.
print(f"\n3. вес: {pay.memory_usage(deep=True).sum() / 1e6:.1f} МБ")

In [ ]:
# Вопрос 4. Одна строка это один платёж? Тогда payment_id уникален.
print("4. payment_id уникален:", pay["payment_id"].is_unique)

# Двойные списания (С3): та же подписка и сумма меньше чем через 20 минут. Ищем среди успешных.
ok = pay[pay["status"] == "success"]
dupes = find_duplicate_payments(ok)
print(f"   двойных списаний: {len(dupes)} на сумму {dupes['amount'].sum():,.0f} руб.".replace(",", " "))

# Вопрос 5. Где пропуски.
print("\n5. пропуски по колонкам:"); print(pay.isna().sum())

# Вопрос 6. Нет ли невозможных значений.
print("\n6. статусы:"); print(pay["status"].value_counts(dropna=False))
print("   сумма <= 0:", (pay["amount"] <= 0).sum())

#### Почему мы считаем их двойными списаниями

Подписка списывается раз в 30 дней. Значит, между двумя успешными платежами одной подписки на одну сумму
в норме проходит около 30 дней. Если проходят минуты, это не продление, а повтор того же списания.
Проверяем это на наших данных, а не верим порогу с семинара на слово.

In [ ]:
# Для каждого успешного платежа: сколько прошло с предыдущего платежа той же подписки на ту же сумму.
s = ok.sort_values(["subscription_id", "amount", "paid_at"])
gap = s.groupby(["subscription_id", "amount"])["paid_at"].diff()   # у первого платежа подписки будет NaT

# Раскладываем промежутки по корзинам: где живут "нормальные" платежи, а где повторы.
bins = pd.to_timedelta(["0s", "1min", "20min", "1D", "25D", "35D", "3650D"])
labels = ["меньше 1 минуты", "1-20 минут", "20 минут - 1 день", "1-25 дней", "25-35 дней (продление)", "больше 35 дней"]
print("Промежуток до предыдущего такого же платежа:")
print(pd.cut(gap.dropna(), bins=bins, labels=labels, right=False).value_counts(sort=False))

# Анализ чувствительности (С3): если ответ не меняется в широком диапазоне порога,
# значит он не зависит от нашего выбора "20 минут".
print("\nПорог -> сколько повторов:")
for m in (1, 5, 15, 20, 60, 180, 1440, 7 * 1440):
    print(f"{m:>6} мин -> {(gap < pd.Timedelta(minutes=m)).sum()}")

In [ ]:
# Как выглядят повторы вблизи: через сколько секунд после первого платежа и сколько подписок задето.
is_dup = gap < pd.Timedelta(minutes=20)
print("медиана промежутка у повторов:", gap[is_dup].median())
print("самый длинный промежуток у повторов:", gap[is_dup].max())
print("подписок с повтором:", s.loc[is_dup, "subscription_id"].nunique())

# Вернули ли эти деньги? Ищем возврат (refunded) по той же подписке в течение 7 дней после повтора.
refunds = pay.loc[pay["status"] == "refunded", ["subscription_id", "paid_at"]]
d = s.loc[is_dup, ["subscription_id", "paid_at"]].merge(refunds, on="subscription_id", suffixes=("", "_ref"))
d = d[(d["paid_at_ref"] >= d["paid_at"]) & (d["paid_at_ref"] < d["paid_at"] + pd.Timedelta(days=7))]
print("повторов, по которым был возврат в течение 7 дней:", d["paid_at"].nunique(), "из", is_dup.sum())

# Пример одной подписки с повтором: все её успешные платежи в периоде.
example = s.loc[is_dup, "subscription_id"].iloc[0]
s.loc[s["subscription_id"] == example, ["paid_at", "amount", "status"]]

In [ ]:
# Вопрос 7. Какой период покрыт и нет ли дыр по дням.
# Сравниваем дни, которые есть в данных, с полным календарём периода.
days_in_data = pd.DatetimeIndex(pay["paid_at"].dt.tz_localize(None).dt.normalize().unique())
all_days = pd.date_range(PERIOD[0], PERIOD[1], freq="D", inclusive="left")   # inclusive="left": без 1 июля
missing = all_days.difference(days_in_data)

print("7. первый и последний платёж:", pay["paid_at"].min(), "...", pay["paid_at"].max())
print("   дней в календаре:", len(all_days), " дней с платежами:", len(days_in_data))
print("   дни без единого платежа:", [d.strftime("%Y-%m-%d") for d in missing])

### 1.3 Какие суммы реально списывались

Цены в справочнике поменялись 1 марта. Но это не значит, что 1 марта все стали платить по-новому.
Смотрим, сколько успешных платежей прошло по каждой сумме в каждом месяце.

In [ ]:
# Месяц платежа по UTC. tz_localize(None) снимает пояс перед to_period: иначе pandas предупреждает,
# что пояс потеряется (Л4, слайд 23). Время при этом не сдвигается: остаются часы UTC.
ok = pay[pay["status"] == "success"].copy()
ok["month"] = ok["paid_at"].dt.tz_localize(None).dt.to_period("M")

# Строки: месяц. Колонки: тариф и сумма. В клетке: сколько платежей.
amounts = ok.groupby(["month", "tariff_id", "amount"]).size().unstack(["tariff_id", "amount"], fill_value=0)
amounts

### 1.4 Подписки

Канал привлечения берём у **владельца** подписки. У части клиентов канала нет: считаем их с `dropna=False`,
чтобы они не пропали молча (Л3, слайд 25).

In [ ]:
subs = q("""
    select s.subscription_id, s.tariff_id, s.started_at, s.ended_at, s.is_trial,
           c.acquisition_channel
    from prime.subscriptions s
    join prime.clients c on c.client_id = s.owner_client_id
""")

expected_subs = q("select count(*) from prime.subscriptions").iloc[0, 0]

print("строк:", len(subs), " ждали:", expected_subs, "->", "ok" if len(subs) == expected_subs else "ВНИМАНИЕ")
print("subscription_id уникален:", subs["subscription_id"].is_unique)
print("\nтипы:"); print(subs.dtypes)
print("\nпропуски:"); print(subs.isna().sum())
# Инвариант двух колонок: подписка не может закончиться раньше, чем началась.
print("\nзакончились раньше начала:", (subs["ended_at"] < subs["started_at"]).sum())
print("\nканал владельца, вместе с пропусками:")
print(subs["acquisition_channel"].value_counts(dropna=False))

### 1.5 Итог приёмки и чистые таблицы

| Что нашли | Что делаем |
|---|---|
| Строка 429 руб. в справочнике перекрывает две соседние, платежей на 429 руб. нет ни одного | убираем строку из справочника |
| Новая цена включилась для всех 1 марта 2026: до марта нет ни одного платежа по новой цене, с марта ни одного по старой | эффект цены меряем на переходе февраль-март |
| 1 189 двойных списаний (повтор через 0-15 минут), возвратов по ним нет | исключаем из выручки |
| 14 и 15 апреля 2026 нет ни одного платежа: дыра в выгрузке | апрель считаем в день только по дням с данными |
| Статусы failed и refunded | в выручку не берём |
| 6 736 подписок без канала владельца | показываем строкой "без канала" |

In [ ]:
# Справочник без кривой строки 429 руб.: ни одного платежа на 429 руб. нет (ячейка 1.3), значит это ошибка ввода.
tariffs_fixed = tariffs[tariffs["monthly_fee"] != 429].copy()

# Проверяем, что пересечений больше нет. Проверка, которая может упасть, и есть настоящая проверка.
a = tariffs_fixed.reset_index(names="row_id")
pairs = a.merge(a, on="tariff_id", suffixes=("_a", "_b"))
n_overlap = ((pairs["row_id_a"] < pairs["row_id_b"])
             & (pairs["valid_from_a"] <= pairs["valid_to_b"])
             & (pairs["valid_from_b"] <= pairs["valid_to_a"])).sum()
assert n_overlap == 0, "в справочнике остались пересечения"
print(tariffs_fixed[["tariff_id", "tariff_name", "monthly_fee", "valid_from", "valid_to"]].to_string(index=False))

In [ ]:
success = pay[pay["status"] == "success"]   # деньги приносят только успешные платежи

# Исключили платежи, чтобы не вести двойной счёт по двойным списаниям за 1 подписку в течение 20 минут.
# (Нормально, если платёж повторяется через 25+ дней. Подтверждено данными: ячейка с корзинами промежутков выше.)
dupes = find_duplicate_payments(success, gap_minutes=20)   # функция возвращает ВТОРОЙ платёж пары, первый остаётся
paid = success[~success.index.isin(dupes.index)].copy()

# Месяц платежа по UTC: понадобится на шаге 2.
paid["month"] = paid["paid_at"].dt.tz_localize(None).dt.to_period("M")

# Контроль: убрали ровно столько строк, сколько нашли дублей, и ни одной лишней.
assert len(paid) == len(success) - len(dupes), "убрали не те строки"
print(f"успешных платежей: {len(success):>9}")
print(f"из них дублей:     {len(dupes):>9}")
print(f"осталось (paid):   {len(paid):>9}")
print(f"выручка до чистки:   {success['amount'].sum():>15,.2f} руб.".replace(",", " "))
print(f"выручка после:       {paid['amount'].sum():>15,.2f} руб.".replace(",", " "))

## 2. Метрики по месяцам и зерно расчёта

*Заполним на шаге 2.*

## 3. Факторное разложение выручки

*Заполним на шаге 3.*

## 4. Проверка гипотезы про отток

*Заполним на шаге 4.*

## 5. Графики

*Заполним на шаге 5.*

## 6. Вывод

*Заполним на шаге 6.*